<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-10-production-deployment/lesson-10.4-caching-strategies/notebooks/exercises-10.4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 10.4 — Caching Strategies
Netsetos GenAI Course v2.0 | Module 10: Production Deployment

Exact + semantic + provider prompt cache, tiering, invalidation, threshold tuning, cost math.


In [ ]:
print('Module 10: Production Deployment')
print('Lesson 10.4: Caching Strategies')


## Ex 1: Why Cache — Latency + Cost Impact


In [ ]:
print('Latency + cost impact of cache hit:')
baseline_latency_ms = 2500
baseline_cost_usd   = 0.003
for tier, lat_ms, cost_ratio in [
    ('no cache',        2500, 1.0),
    ('provider prompt', 1100, 0.1),  # Anthropic 90% read discount
    ('redis semantic',    12, 0.0),
    ('redis exact',        2, 0.0),
    ('in-process LRU',     0.005, 0.0),
]:
    x = baseline_latency_ms / max(lat_ms, 0.001)
    print(f'  {tier:20s} lat={lat_ms:>8.3f}ms ({x:>7.1f}x faster) cost={cost_ratio*baseline_cost_usd:.5f}')


## Ex 2: 4-Tier Architecture


In [ ]:
print('4-tier cache order (each tier catches what the next would miss):')
for tier, where, latency, hit_rate in [
    ('L1','In-process LRU','~5 us','15-25%'),
    ('L2','Redis exact (SHA256)','~2 ms','20-35%'),
    ('L3','Redis semantic (embed)','~12 ms','15-30% on top of exact'),
    ('L4','Provider prompt cache','server-side','60-90% on prefix'),
]:
    print(f'  {tier}: {where:28s} | lat={latency:14s} | hit={hit_rate}')


## Ex 3: Provider Prompt Cache Pricing


In [ ]:
print('Provider prompt cache pricing (2026, illustrative):')
for prov, write, read in [
    ('Anthropic 5min',      '1.25x base',  '0.1x base (90% off)'),
    ('Anthropic 1h',        '2x base',     '0.1x base'),
    ('OpenAI gpt-4o auto',  'no penalty',  '0.5x input (50% off)'),
    ('OpenAI gpt-4o-mini',  'no penalty',  '0.25x input (75% off)'),
    ('Gemini implicit',     'no penalty',  '0.25x input (implicit cache)'),
]:
    print(f'  {prov:22s}: write {write:14s} | read {read}')


## Ex 4: Hit Rate vs Cost Savings Math


In [ ]:
calls_per_day = 100_000
cost_per_call = 0.015  # $ (avg mid-tier)
print(f'Workload: {calls_per_day:,}/day @ ${cost_per_call}/call = ${calls_per_day*cost_per_call:,.0f}/day baseline')
print()
for hit in [0.20, 0.40, 0.60, 0.80]:
    saved = calls_per_day * hit * cost_per_call
    monthly = saved * 30
    yearly = saved * 365
    print(f'  hit={hit:.0%} -> daily save ${saved:,.0f} | month ${monthly:,.0f} | year ${yearly:,.0f}')


## Ex 5: Similarity Threshold Trade-off


In [ ]:
print('Semantic cache threshold trade-off (illustrative labeled set):')
for t, precision, recall, note in [
    (0.88, 0.72, 0.94, 'aggressive - wrong-answer risk'),
    (0.92, 0.84, 0.88, 'balanced but risky'),
    (0.95, 0.94, 0.71, 'elbow - typical production choice'),
    (0.97, 0.98, 0.52, 'conservative - fewer hits'),
    (0.99, 1.00, 0.18, 'too tight - barely helps'),
]:
    print(f'  t={t:.2f} | precision={precision:.2f} recall={recall:.2f} | {note}')
print()
print('Tune per domain; healthcare/legal demand 0.97+. Re-tune after embedding model changes.')


## Ex 6: Cache Key Design


In [ ]:
print('Cache key must include ALL of:')
for part, why in [
    ('model',             'different models give different answers'),
    ('system prompt hash','persona/rules changes -> different answer'),
    ('user prompt',       'the actual question'),
    ('tools schema hash', 'available tools change the answer'),
    ('tenant_id',         'tenant-specific context/config'),
    ('locale (optional)', 'output language differs'),
]:
    print(f'  {part:22s}: {why}')
print()
print('Normalize: strip whitespace, lowercase, sort tools before hashing.')
print('NEVER include timestamps, request ids, or user_id in the key.')


## Ex 7: What NOT to Cache


In [ ]:
print('Skip the cache when:')
for case, why in [
    ('Mutating tool calls (writes)','Side effect must re-execute'),
    ('Personalized completions','Context per user; wrong-answer risk'),
    ('PII-laden prompts','Legal retention boundary'),
    ('Time-sensitive (stock, weather)','Stale answer = wrong'),
    ('Adversarial / red-team calls','Must hit real model + audit trail'),
]:
    print(f'  {case:34s}: {why}')
print()
print("Mark with cache_skip header at the gateway; it's the contract.")


---
## Recap
4-tier cache: in-process LRU + Redis exact + Redis semantic + provider prompt cache.
Anthropic: 90% read savings @ 0.1x. OpenAI: 50-75% auto.
Key = hash(model + system + user + tools + tenant), normalized.
Threshold 0.95 typical; 0.97+ for regulated domains. NEVER cache mutations, PII, time-sensitive.
Realistic workload: 70%+ cost cut on $1.5M/yr bill = $1M saved.
